# Notebook 03: Feature Engineering and Leakage Review

This notebook converts the cleaned diabetes readmission dataset into a model-ready dataset.

The goal is to:

- define prediction timing
- engineer model-ready features
- review leakage risk
- document excluded fields
- preserve `patient_nbr` for patient-aware splitting
- export the model-ready dataset

This notebook does not train models.

## 1. Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

## 2. Project Paths

In [2]:
def find_project_root(start_path=None):
    """
    Finds the project root by walking upward until the cleaned dataset path is found.
    This avoids relying on the current working directory being exactly the project root
    or the notebooks folder.
    """
    start_path = Path.cwd() if start_path is None else Path(start_path)

    for path in [start_path, *start_path.parents]:
        expected_file = path / "data" / "processed" / "diabetes_readmission_cleaned.csv"
        if expected_file.exists():
            return path

    raise FileNotFoundError(
        "Could not locate project root. Expected to find "
        "'data/processed/diabetes_readmission_cleaned.csv' in this folder or a parent folder."
    )


PROJECT_ROOT = find_project_root()

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS = PROJECT_ROOT / "outputs"
MODEL_RESULTS = OUTPUTS / "model_results"

for path in [DATA_PROCESSED, OUTPUTS, MODEL_RESULTS]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root detected successfully.")
print("Project folder:", PROJECT_ROOT.name)

Project root detected successfully.
Project folder: 03_risk_stratification_intervention_prioritization


## 3. Load and Validate Cleaned Data

In [3]:
cleaned_path = DATA_PROCESSED / "diabetes_readmission_cleaned.csv"

assert cleaned_path.exists(), f"Missing cleaned dataset: {cleaned_path}"

df = pd.read_csv(cleaned_path, low_memory=False)

# Validate that pseudo-null values from the raw dataset were converted during cleaning.
pseudo_null_values = ["?", "None", "none", "NULL", "null", "nan", "NaN", ""]

columns_to_check_for_pseudo_nulls = [
    "A1Cresult",
    "max_glu_serum",
    "weight",
    "payer_code",
    "medical_specialty"
]

columns_to_check_for_pseudo_nulls = [
    col for col in columns_to_check_for_pseudo_nulls
    if col in df.columns
]

pseudo_null_check = {}

for col in columns_to_check_for_pseudo_nulls:
    pseudo_null_check[col] = df[col].astype("string").isin(pseudo_null_values).sum()

pseudo_null_check = pd.Series(pseudo_null_check, name="pseudo_null_count")

display(pseudo_null_check)

assert pseudo_null_check.sum() == 0, (
    "Pseudo-null values remain in the cleaned dataset. "
    "Review Notebook 01 missing-value cleaning before feature engineering."
)

print("Cleaned dataset shape:", df.shape)
display(df.head())

A1Cresult            0
max_glu_serum        0
weight               0
payer_code           0
medical_specialty    0
Name: pseudo_null_count, dtype: int64

Cleaned dataset shape: (101766, 51)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,readmitted_30d
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO,0
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30,0
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO,0
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO,0
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,0


In [4]:
required_columns = [
    "encounter_id",
    "patient_nbr",
    "readmitted",
    "readmitted_30d"
]

missing_required = [col for col in required_columns if col not in df.columns]

assert len(missing_required) == 0, f"Missing required columns: {missing_required}"
assert set(df["readmitted_30d"].dropna().unique()) <= {0, 1}, "Target must be binary."

print("Required columns found.")
print(f"Baseline 30-day readmission rate: {df['readmitted_30d'].mean():.2%}")
print(f"Total encounters: {len(df):,}")
print(f"Unique encounters: {df['encounter_id'].nunique():,}")
print(f"Unique patients: {df['patient_nbr'].nunique():,}")
print(f"Repeated patient rows: {len(df) - df['patient_nbr'].nunique():,}")

Required columns found.
Baseline 30-day readmission rate: 11.16%
Total encounters: 101,766
Unique encounters: 101,766
Unique patients: 71,518
Repeated patient rows: 30,248


### Interpretation

The cleaned dataset loaded successfully.

The project remains encounter-level because each row represents a hospital encounter.

However, the dataset contains repeated patients, so later model evaluation should use patient-aware splitting. This prevents encounters from the same patient appearing in both training and testing data.

## 4. Prediction Timing and Modeling Unit

### Prediction Timing

This project uses a near-discharge prediction framing.

The model is assumed to generate a 30-day readmission risk score at or near discharge, before post-discharge outreach is assigned.

This means some fields are more defensible than they would be under an admission-time model.

Examples:

- `time_in_hospital` may be known by discharge
- `discharge_disposition_id` may be known by discharge
- medication changes during the encounter may be known by discharge

### Modeling Unit

The unit of prediction is the encounter.

The unit of train/test separation should be the patient.

This means the model predicts encounter-level risk, but evaluation should keep all encounters from the same patient either in train or test, not both.

## 5. Patient-Aware Split Policy

In [5]:
patient_encounter_counts = (
    df["patient_nbr"]
    .value_counts()
    .rename_axis("patient_nbr")
    .reset_index(name="encounter_count")
)

repeated_patient_summary = pd.DataFrame({
    "metric": [
        "total_encounters",
        "unique_patients",
        "patients_with_multiple_encounters",
        "repeated_patient_rows"
    ],
    "value": [
        len(df),
        df["patient_nbr"].nunique(),
        (patient_encounter_counts["encounter_count"] > 1).sum(),
        len(df) - df["patient_nbr"].nunique()
    ]
})

display(repeated_patient_summary)
display(patient_encounter_counts["encounter_count"].describe())

,metric,value
0,total_encounters,101766
1,unique_patients,71518
2,patients_with_multiple_encounters,16773
3,repeated_patient_rows,30248


count    71518.000000
mean         1.422942
std          1.090740
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         40.000000
Name: encounter_count, dtype: float64

### Interpretation

A random row-level split is not recommended for final modeling.

Reason:

The same patient may appear in both training and test data. This creates patient-level leakage and may overstate model performance.

The project will use:

- `GroupShuffleSplit` for the main train/test split
- `GroupKFold` if cross-validation is used later

The grouping variable will be `patient_nbr`.

This is the correct compromise:

- prediction unit: encounter
- split unit: patient

## 6. Start Feature Engineering Dataset

In [6]:
# Create a working copy so the cleaned dataset remains unchanged.
df_fe = df.copy()

print("Feature engineering dataset shape:", df_fe.shape)

Feature engineering dataset shape: (101766, 51)


In [7]:
tracking_columns = [
    "encounter_id",
    "patient_nbr"
]

target_column = "readmitted_30d"

original_target_columns = [
    "readmitted"
]

print("Tracking columns:", tracking_columns)
print("Target column:", target_column)
print("Original target columns excluded from modeling:", original_target_columns)

Tracking columns: ['encounter_id', 'patient_nbr']
Target column: readmitted_30d
Original target columns excluded from modeling: ['readmitted']


### Interpretation

`encounter_id` and `patient_nbr` are retained for tracking, auditing, and patient-aware splitting.

They should not be used as predictive features.

The original `readmitted` column is excluded from modeling because it directly defines the binary target.

## 7. Age Features

In [8]:
age_midpoint_map = {
    "[0-10)": 5,
    "[10-20)": 15,
    "[20-30)": 25,
    "[30-40)": 35,
    "[40-50)": 45,
    "[50-60)": 55,
    "[60-70)": 65,
    "[70-80)": 75,
    "[80-90)": 85,
    "[90-100)": 95
}

age_order_map = {
    age_group: order
    for order, age_group in enumerate(age_midpoint_map.keys(), start=1)
}

# Age is stored as interval strings, so create numeric versions for modeling.
df_fe["age_midpoint"] = df_fe["age"].map(age_midpoint_map)
df_fe["age_order"] = df_fe["age"].map(age_order_map)

display(
    df_fe[["age", "age_midpoint", "age_order"]]
    .drop_duplicates()
    .sort_values("age_order")
)

,age,age_midpoint,age_order
0,[0-10),5,1
1,[10-20),15,2
2,[20-30),25,3
3,[30-40),35,4
4,[40-50),45,5
5,[50-60),55,6
6,[60-70),65,7
7,[70-80),75,8
8,[80-90),85,9
9,[90-100),95,10


### Interpretation

The original `age` field is an ordered age band, not an exact numeric age.

`age_midpoint` provides a simple numeric approximation.

`age_order` preserves age-band ordering without pretending exact age is known.

## 8. Prior Utilization Features

In [9]:
# Prior utilization variables summarize healthcare use before the current encounter.
df_fe["prior_outpatient_flag"] = np.where(df_fe["number_outpatient"] > 0, 1, 0)
df_fe["prior_emergency_flag"] = np.where(df_fe["number_emergency"] > 0, 1, 0)
df_fe["prior_inpatient_flag"] = np.where(df_fe["number_inpatient"] > 0, 1, 0)

df_fe["total_prior_visits"] = (
    df_fe["number_outpatient"]
    + df_fe["number_emergency"]
    + df_fe["number_inpatient"]
)

df_fe["high_prior_utilization_flag"] = np.where(df_fe["total_prior_visits"] >= 4, 1, 0)

display(
    df_fe[
        [
            "number_outpatient",
            "number_emergency",
            "number_inpatient",
            "total_prior_visits",
            "high_prior_utilization_flag"
        ]
    ].describe()
)

,number_outpatient,number_emergency,number_inpatient,total_prior_visits,high_prior_utilization_flag
count,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000
mean,0.369357,0.197836,0.635566,1.202759,0.098569
std,1.267265,0.930472,1.262863,2.291781,0.298084
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,1.000000,2.000000,0.000000
max,42.000000,76.000000,21.000000,80.000000,1.000000


In [10]:
df_fe["total_prior_visits_group"] = pd.cut(
    df_fe["total_prior_visits"],
    bins=[-1, 0, 1, 3, 10, np.inf],
    labels=["0", "1", "2-3", "4-10", "11+"]
)

display(
    df_fe["total_prior_visits_group"]
    .value_counts(dropna=False)
    .rename_axis("total_prior_visits_group")
    .reset_index(name="count")
)

,total_prior_visits_group,count
0,0,55828
1,1,19941
2,2-3,15966
3,4-10,9041
4,11+,990


### Interpretation

Prior utilization is useful because it captures historical healthcare use before the current encounter.

These features are generally safer than future-looking fields.

However, they should still be described as predictors of risk, not causes of readmission.

## 9. Encounter Complexity Features

In [11]:
# These features summarize encounter-level complexity.
df_fe["long_stay_flag"] = np.where(df_fe["time_in_hospital"] >= 7, 1, 0)

df_fe["time_in_hospital_group"] = pd.cut(
    df_fe["time_in_hospital"],
    bins=[0, 2, 4, 7, 14],
    labels=["1-2 days", "3-4 days", "5-7 days", "8-14 days"],
    include_lowest=True
)

df_fe["diagnosis_count_group"] = pd.cut(
    df_fe["number_diagnoses"],
    bins=[0, 3, 6, 9, np.inf],
    labels=["1-3", "4-6", "7-9", "10+"],
    include_lowest=True
)

df_fe["medication_count_group"] = pd.cut(
    df_fe["num_medications"],
    bins=[0, 10, 20, 30, np.inf],
    labels=["1-10", "11-20", "21-30", "31+"],
    include_lowest=True
)

display(
    df_fe[
        [
            "time_in_hospital",
            "long_stay_flag",
            "time_in_hospital_group",
            "number_diagnoses",
            "diagnosis_count_group",
            "num_medications",
            "medication_count_group"
        ]
    ].head()
)

,time_in_hospital,long_stay_flag,time_in_hospital_group,number_diagnoses,diagnosis_count_group,num_medications,medication_count_group
0,1,0,1-2 days,1,1-3,1,1-10
1,3,0,3-4 days,9,7-9,18,11-20
2,2,0,1-2 days,6,4-6,13,11-20
3,2,0,1-2 days,7,7-9,16,11-20
4,1,0,1-2 days,5,4-6,8,1-10


### Interpretation

These features summarize encounter complexity.

They are defensible under the near-discharge framing because they are known by the end of the encounter.

They would be less defensible under an admission-time prediction framing.

## 10. Diagnosis Code Features

In [12]:
def categorize_diagnosis_code(code):
    """
    Groups ICD-9 diagnosis codes into broad clinical categories.

    This is a simplified grouping for portfolio modeling.
    It avoids feeding raw high-cardinality diagnosis codes directly into the model.
    """
    if pd.isna(code):
        return "Missing"

    code = str(code).strip()

    if code.startswith("V"):
        return "Supplementary_V"
    if code.startswith("E"):
        return "Supplementary_E"

    try:
        numeric_code = float(code)
    except ValueError:
        return "Other"

    if 390 <= numeric_code <= 459 or numeric_code == 785:
        return "Circulatory"
    elif 460 <= numeric_code <= 519 or numeric_code == 786:
        return "Respiratory"
    elif 520 <= numeric_code <= 579 or numeric_code == 787:
        return "Digestive"
    elif 250 <= numeric_code < 251:
        return "Diabetes"
    elif 800 <= numeric_code <= 999:
        return "Injury"
    elif 710 <= numeric_code <= 739:
        return "Musculoskeletal"
    elif 580 <= numeric_code <= 629 or numeric_code == 788:
        return "Genitourinary"
    elif 140 <= numeric_code <= 239:
        return "Neoplasms"
    else:
        return "Other"

In [13]:
diagnosis_columns = ["diag_1", "diag_2", "diag_3"]

for col in diagnosis_columns:
    if col in df_fe.columns:
        df_fe[f"{col}_category"] = df_fe[col].apply(categorize_diagnosis_code)

diagnosis_category_cols = [
    f"{col}_category"
    for col in diagnosis_columns
    if f"{col}_category" in df_fe.columns
]

for col in diagnosis_category_cols:
    print(f"\n{col}")
    display(
        df_fe[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="count")
        .head(15)
    )


diag_1_category


,diag_1_category,count
0,Circulatory,30437
1,Other,16527
2,Respiratory,14423
3,Digestive,9475
4,Diabetes,8757
5,Injury,6974
6,Genitourinary,5117
7,Musculoskeletal,4957
8,Neoplasms,3433
9,Supplementary_V,1644



diag_2_category


,diag_2_category,count
0,Circulatory,31881
1,Other,24017
2,Diabetes,12794
3,Respiratory,10895
4,Genitourinary,8376
5,Digestive,4170
6,Neoplasms,2547
7,Injury,2428
8,Supplementary_V,1805
9,Musculoskeletal,1764



diag_3_category


,diag_3_category,count
0,Circulatory,30306
1,Other,24137
2,Diabetes,17157
3,Respiratory,7358
4,Genitourinary,6680
5,Digestive,3930
6,Supplementary_V,3814
7,Injury,1946
8,Musculoskeletal,1915
9,Neoplasms,1856


### Interpretation

Raw diagnosis codes are high-cardinality and difficult to interpret directly.

This notebook creates broad diagnosis categories instead of using raw diagnosis codes in the first model-ready dataset.

This is a practical first-pass grouping, not a perfect clinical classification system.

## 11. Medication Features

In [14]:
medication_columns = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide",
    "glimepiride", "acetohexamide", "glipizide", "glyburide",
    "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose",
    "miglitol", "troglitazone", "tolazamide", "examide",
    "citoglipton", "insulin", "glyburide-metformin",
    "glipizide-metformin", "glimepiride-pioglitazone",
    "metformin-rosiglitazone", "metformin-pioglitazone"
]

medication_columns = [col for col in medication_columns if col in df_fe.columns]

# Values such as Up and Down indicate medication changes during the encounter.
df_fe["medication_change_count"] = df_fe[medication_columns].isin(["Up", "Down"]).sum(axis=1)
df_fe["any_medication_change_flag"] = np.where(df_fe["medication_change_count"] > 0, 1, 0)

# These features simplify common diabetes-medication indicators.
df_fe["insulin_flag"] = np.where(df_fe["insulin"].isin(["Steady", "Up", "Down"]), 1, 0)
df_fe["diabetes_med_flag"] = np.where(df_fe["diabetesMed"] == "Yes", 1, 0)
df_fe["diabetes_med_changed_flag"] = np.where(df_fe["change"] == "Ch", 1, 0)

display(
    df_fe[
        [
            "medication_change_count",
            "any_medication_change_flag",
            "insulin",
            "insulin_flag",
            "diabetesMed",
            "diabetes_med_flag",
            "change",
            "diabetes_med_changed_flag"
        ]
    ].head()
)

,medication_change_count,any_medication_change_flag,insulin,insulin_flag,diabetesMed,diabetes_med_flag,change,diabetes_med_changed_flag
0,0,0,No,0,No,0,No,0
1,1,1,Up,1,Yes,1,Ch,1
2,0,0,No,0,Yes,1,No,0
3,1,1,Up,1,Yes,1,Ch,1
4,0,0,Steady,1,Yes,1,Ch,1


In [15]:
medication_feature_summary = pd.DataFrame({
    "metric": [
        "mean_medication_change_count",
        "any_medication_change_rate",
        "insulin_flag_rate",
        "diabetes_med_flag_rate",
        "diabetes_med_changed_rate"
    ],
    "value": [
        df_fe["medication_change_count"].mean(),
        df_fe["any_medication_change_flag"].mean(),
        df_fe["insulin_flag"].mean(),
        df_fe["diabetes_med_flag"].mean(),
        df_fe["diabetes_med_changed_flag"].mean()
    ]
})

display(medication_feature_summary)

,metric,value
0,mean_medication_change_count,0.287444
1,any_medication_change_rate,0.272223
2,insulin_flag_rate,0.534393
3,diabetes_med_flag_rate,0.770031
4,diabetes_med_changed_rate,0.461952


### Interpretation

Medication-related variables may be available during or by the end of the encounter.

Under the near-discharge framing, they may be valid candidate predictors.

However, medication changes may also reflect clinical response during hospitalization, so they should be documented in the leakage review instead of used blindly.

## 12. Missingness Indicator Features

In [16]:
# Missingness indicators preserve whether important fields were unavailable.
missing_indicator_candidates = [
    "race",
    "payer_code",
    "medical_specialty",
    "weight"
]

for col in missing_indicator_candidates:
    if col in df_fe.columns:
        df_fe[f"{col}_missing_flag"] = np.where(df_fe[col].isna(), 1, 0)

missing_indicator_cols = [
    f"{col}_missing_flag"
    for col in missing_indicator_candidates
    if f"{col}_missing_flag" in df_fe.columns
]

missing_indicator_summary = (
    df_fe[missing_indicator_cols]
    .mean()
    .reset_index()
    .rename(columns={"index": "missing_indicator", 0: "missing_rate"})
)

display(missing_indicator_summary)

,missing_indicator,missing_rate
0,race_missing_flag,0.022336
1,payer_code_missing_flag,0.395574
2,medical_specialty_missing_flag,0.490822
3,weight_missing_flag,0.968585


### Interpretation

Missingness indicators may be useful when missingness itself carries information.

For example, missing `medical_specialty` or missing `payer_code` may reflect documentation patterns.

However, missingness can also reflect data quality problems, so these features should be interpreted carefully.

## 13. Rare Category Grouping

In [17]:
def group_rare_categories(series, min_count=1000, missing_label="Missing", rare_label="Rare"):
    """
    Groups low-frequency categories into a shared Rare category.

    This reduces sparse one-hot encoded columns later.
    """
    filled = series.fillna(missing_label).astype(str)
    value_counts = filled.value_counts()
    common_values = value_counts[value_counts >= min_count].index

    return np.where(filled.isin(common_values), filled, rare_label)

In [18]:
if "medical_specialty" in df_fe.columns:
    df_fe["medical_specialty_grouped"] = group_rare_categories(
        df_fe["medical_specialty"],
        min_count=1000
    )

if "payer_code" in df_fe.columns:
    df_fe["payer_code_grouped"] = group_rare_categories(
        df_fe["payer_code"],
        min_count=1000
    )

grouped_category_cols = [
    col for col in ["medical_specialty_grouped", "payer_code_grouped"]
    if col in df_fe.columns
]

for col in grouped_category_cols:
    print(f"\n{col}")
    display(
        df_fe[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="count")
        .head(20)
    )


medical_specialty_grouped


,medical_specialty_grouped,count
0,Missing,49949
1,InternalMedicine,14635
2,Rare,8340
3,Emergency/Trauma,7565
4,Family/GeneralPractice,7440
5,Cardiology,5352
6,Surgery-General,3099
7,Nephrology,1613
8,Orthopedics,1400
9,Orthopedics-Reconstructive,1233



payer_code_grouped


,payer_code_grouped,count
0,Missing,40256
1,MC,32439
2,HM,6274
3,SP,5007
4,BC,4655
5,MD,3532
6,CP,2533
7,UN,2448
8,CM,1937
9,Rare,1652


`payer_code_grouped` is created for review but excluded from the first primary feature set because payer information may proxy insurance or socioeconomic patterns. The missingness indicator is retained for later evaluation.

### Interpretation

High-cardinality categorical fields can create many sparse encoded features.

Rare category grouping keeps the feature space more stable and easier to interpret.

The raw versions of these columns should not be used together with grouped versions in the first model-ready dataset.

## 14. Operational ID Fields as Categorical Features

In [19]:
operational_id_columns = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

for col in operational_id_columns:
    if col in df_fe.columns:
        # These are category codes, not continuous numeric measures.
        df_fe[f"{col}_cat"] = df_fe[col].astype("Int64").astype(str)

operational_cat_cols = [
    f"{col}_cat"
    for col in operational_id_columns
    if f"{col}_cat" in df_fe.columns
]

display(df_fe[operational_cat_cols].head())

,admission_type_id_cat,discharge_disposition_id_cat,admission_source_id_cat
0,6,25,1
1,1,1,7
2,1,1,7
3,1,1,7
4,1,1,7


### Interpretation

Admission type, discharge disposition, and admission source are coded as numbers, but they represent categories.

Creating categorical versions prevents the model from treating these IDs as continuous numeric variables.

These fields still require leakage review because their acceptability depends on prediction timing.

## 15. Leakage Review

In [20]:
leakage_review = pd.DataFrame([
    {
        "field_or_feature": "encounter_id",
        "leakage_risk": "high",
        "decision": "exclude_from_model_keep_for_tracking",
        "rationale": "Identifier with no predictive meaning; could allow record memorization."
    },
    {
        "field_or_feature": "patient_nbr",
        "leakage_risk": "high",
        "decision": "exclude_from_model_keep_for_group_split",
        "rationale": "Patient identifier should not be used as a feature but is needed for patient-aware splitting."
    },
    {
        "field_or_feature": "readmitted",
        "leakage_risk": "direct_target_leakage",
        "decision": "exclude_from_model",
        "rationale": "Original target column directly defines readmitted_30d."
    },
    {
        "field_or_feature": "readmitted_30d",
        "leakage_risk": "target",
        "decision": "use_as_target_only",
        "rationale": "Binary target for 30-day readmission prediction."
    },
    {
        "field_or_feature": "time_in_hospital",
        "leakage_risk": "timing_dependent",
        "decision": "include_under_near_discharge_framing",
        "rationale": "Known by discharge; would be excluded for admission-time prediction."
    },
    {
        "field_or_feature": "discharge_disposition_id_cat",
        "leakage_risk": "timing_dependent",
        "decision": "include_under_near_discharge_framing",
        "rationale": "Likely known at discharge; may encode post-acute care status."
    },
    {
        "field_or_feature": "admission_type_id_cat",
        "leakage_risk": "low_to_moderate",
        "decision": "include",
        "rationale": "Encounter context field generally available early in the encounter."
    },
    {
        "field_or_feature": "admission_source_id_cat",
        "leakage_risk": "low_to_moderate",
        "decision": "include",
        "rationale": "Encounter source field generally available early in the encounter."
    },
    {
        "field_or_feature": "diag_1_category / diag_2_category / diag_3_category",
        "leakage_risk": "timing_dependent",
        "decision": "include_broad_categories",
        "rationale": "Diagnosis information may be available by discharge; broad categories reduce high-cardinality noise."
    },
    {
        "field_or_feature": "medication_change_count",
        "leakage_risk": "timing_dependent",
        "decision": "include_under_near_discharge_framing",
        "rationale": "Medication changes occur during the encounter and may be known by discharge."
    },
    {
        "field_or_feature": "race",
        "leakage_risk": "sensitive_demographic",
        "decision": "exclude_from_primary_model_keep_for_subgroup_review",
        "rationale": "Sensitive demographic field; retained for fairness and subgroup review."
    },
    {
        "field_or_feature": "gender",
        "leakage_risk": "sensitive_demographic",
        "decision": "exclude_from_primary_model_keep_for_subgroup_review",
        "rationale": "Sensitive demographic field; retained for fairness and subgroup review."
    },
    {
        "field_or_feature": "age_midpoint / age_order",
        "leakage_risk": "sensitive_or_clinical_demographic",
        "decision": "include_age_features_in_primary_model",
        "rationale": "Age is clinically relevant, but subgroup performance must be reviewed later."
    },
    {
        "field_or_feature": "payer_code",
        "leakage_risk": "proxy_and_missingness_risk",
        "decision": "exclude_raw_use_grouped_or_missing_indicator_only",
        "rationale": "May proxy insurance or socioeconomic patterns and has missingness."
    },
    {
        "field_or_feature": "medical_specialty",
        "leakage_risk": "high_missingness_and_high_cardinality",
        "decision": "exclude_raw_use_grouped_version",
        "rationale": "Raw field is sparse and high-cardinality; grouped version is more stable."
    },
    {
        "field_or_feature": "weight",
        "leakage_risk": "extreme_missingness",
        "decision": "exclude_raw_keep_missing_indicator",
        "rationale": "Very high missingness makes the raw feature unreliable."
    },
    {
    "field_or_feature": "A1Cresult",
    "leakage_risk": "high_missingness_and_timing_dependent",
    "decision": "exclude_raw_or_use_tested_flag_for_first_model",
    "rationale": "High missingness; may reflect whether A1C testing occurred during the encounter. Use cautiously under near-discharge framing."
    },
    {
        "field_or_feature": "max_glu_serum",
        "leakage_risk": "extreme_missingness_and_timing_dependent",
        "decision": "exclude_raw_or_use_tested_flag_for_first_model",
        "rationale": "Very high missingness; may reflect whether glucose testing occurred during the encounter. Use cautiously under near-discharge framing."
    }
])

display(leakage_review)

,field_or_feature,leakage_risk,decision,rationale
0,encounter_id,high,exclude_from_model_keep_for_tracking,Identifier with no predictive meaning; could a...
1,patient_nbr,high,exclude_from_model_keep_for_group_split,Patient identifier should not be used as a fea...
2,readmitted,direct_target_leakage,exclude_from_model,Original target column directly defines readmi...
3,readmitted_30d,target,use_as_target_only,Binary target for 30-day readmission prediction.
4,time_in_hospital,timing_dependent,include_under_near_discharge_framing,Known by discharge; would be excluded for admi...
5,discharge_disposition_id_cat,timing_dependent,include_under_near_discharge_framing,Likely known at discharge; may encode post-acu...
6,admission_type_id_cat,low_to_moderate,include,Encounter context field generally available ea...
7,admission_source_id_cat,low_to_moderate,include,Encounter source field generally available ear...
8,diag_1_category / diag_2_category / diag_3_cat...,timing_dependent,include_broad_categories,Diagnosis information may be available by disc...
9,medication_change_count,timing_dependent,include_under_near_discharge_framing,Medication changes occur during the encounter ...


### Interpretation

The most important leakage decision is prediction timing.

This project uses a near-discharge prediction framing, so some fields are acceptable that would not be acceptable for admission-time prediction.

Identifiers and target-derived columns are excluded.

Sensitive demographic fields are retained for subgroup review, not automatically used in the primary model.

In [21]:
# Lab-result fields have very high missingness.
# Instead of treating missing values as ordinary blanks, create test-performed indicators.
lab_result_columns = ["A1Cresult", "max_glu_serum"]

for col in lab_result_columns:
    if col in df_fe.columns:
        df_fe[f"{col}_tested_flag"] = np.where(df_fe[col].notna(), 1, 0)

`A1Cresult` and `max_glu_serum` have very high missingness. In this first model-ready feature set, the raw categorical lab-result fields are excluded to avoid treating extremely sparse values as regular predictors.

Instead, this notebook creates tested/not-tested flags:

- `A1Cresult_tested_flag`
- `max_glu_serum_tested_flag`

These flags preserve whether the lab result was available without carrying the highly sparse raw result categories into the first modeling dataset.

## 16. Define Model-Ready Feature Set

In [22]:
numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "age_midpoint",
    "age_order",
    "total_prior_visits",
    "medication_change_count"
]

binary_features = [
    "A1Cresult_tested_flag",
    "max_glu_serum_tested_flag",
    "prior_outpatient_flag",
    "prior_emergency_flag",
    "prior_inpatient_flag",
    "high_prior_utilization_flag",
    "long_stay_flag",
    "any_medication_change_flag",
    "insulin_flag",
    "diabetes_med_flag",
    "diabetes_med_changed_flag"
]

categorical_features = [
    "time_in_hospital_group",
    "total_prior_visits_group",
    "diagnosis_count_group",
    "medication_count_group",
    "admission_type_id_cat",
    "discharge_disposition_id_cat",
    "admission_source_id_cat",
    "diag_1_category",
    "diag_2_category",
    "diag_3_category",
    "medical_specialty_grouped"
]

# Add missingness indicators if they were created.
missing_indicator_features = [
    col for col in df_fe.columns
    if col.endswith("_missing_flag")
]

# Keep only columns that actually exist in the local dataset.
numeric_features = [col for col in numeric_features if col in df_fe.columns]
binary_features = [col for col in binary_features if col in df_fe.columns]
categorical_features = [col for col in categorical_features if col in df_fe.columns]

primary_model_features = (
    numeric_features
    + binary_features
    + categorical_features
    + missing_indicator_features
)

print("Numeric features:", len(numeric_features))
print("Binary features:", len(binary_features))
print("Categorical features:", len(categorical_features))
print("Missing indicator features:", len(missing_indicator_features))
print("Total primary model features:", len(primary_model_features))

Numeric features: 12
Binary features: 11
Categorical features: 11
Missing indicator features: 4
Total primary model features: 38


In [23]:
feature_list = pd.DataFrame({
    "feature": primary_model_features,
    "feature_type": (
        ["numeric"] * len(numeric_features)
        + ["binary"] * len(binary_features)
        + ["categorical"] * len(categorical_features)
        + ["missing_indicator"] * len(missing_indicator_features)
    )
})

display(feature_list)

,feature,feature_type
0,time_in_hospital,numeric
1,num_lab_procedures,numeric
2,num_procedures,numeric
3,num_medications,numeric
4,number_outpatient,numeric
5,number_emergency,numeric
6,number_inpatient,numeric
7,number_diagnoses,numeric
8,age_midpoint,numeric
9,age_order,numeric


### Interpretation

The primary model feature list excludes:

- identifiers
- original target column
- raw high-cardinality diagnosis codes
- raw high-missingness weight
- raw race and gender
- raw patient identifier

The dataset still retains tracking and subgroup-review columns outside the predictive feature list.

## 17. Create Model-Ready Dataset

In [24]:
subgroup_review_columns = [
    "race",
    "gender",
    "age"
]

subgroup_review_columns = [
    col for col in subgroup_review_columns
    if col in df_fe.columns
]

model_ready_columns = (
    tracking_columns
    + [target_column]
    + subgroup_review_columns
    + primary_model_features
)

# Remove accidental duplicates while preserving order.
model_ready_columns = list(dict.fromkeys(model_ready_columns))

df_model_ready = df_fe[model_ready_columns].copy()

# Normalize feature dtypes before export.
# This dataset is modeling-input-ready, but not yet encoded, imputed, scaled, or split.
for col in numeric_features:
    if col in df_model_ready.columns:
        df_model_ready[col] = pd.to_numeric(df_model_ready[col], errors="coerce")

for col in binary_features + missing_indicator_features:
    if col in df_model_ready.columns:
        df_model_ready[col] = df_model_ready[col].astype("int8")

for col in categorical_features + subgroup_review_columns:
    if col in df_model_ready.columns:
        df_model_ready[col] = df_model_ready[col].astype("string")

dtype_review = pd.DataFrame({
    "column": df_model_ready.columns,
    "dtype": df_model_ready.dtypes.astype(str).values
})

display(dtype_review)

print("Model-ready dataset shape:", df_model_ready.shape)
display(df_model_ready.head())

,column,dtype
0,encounter_id,int64
1,patient_nbr,int64
2,readmitted_30d,int64
3,race,string
4,gender,string
5,age,string
6,time_in_hospital,int64
7,num_lab_procedures,int64
8,num_procedures,int64
9,num_medications,int64


Model-ready dataset shape: (101766, 44)


,encounter_id,patient_nbr,readmitted_30d,race,gender,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,age_midpoint,age_order,total_prior_visits,medication_change_count,A1Cresult_tested_flag,max_glu_serum_tested_flag,prior_outpatient_flag,prior_emergency_flag,prior_inpatient_flag,high_prior_utilization_flag,long_stay_flag,any_medication_change_flag,insulin_flag,diabetes_med_flag,diabetes_med_changed_flag,time_in_hospital_group,total_prior_visits_group,diagnosis_count_group,medication_count_group,admission_type_id_cat,discharge_disposition_id_cat,admission_source_id_cat,diag_1_category,diag_2_category,diag_3_category,medical_specialty_grouped,race_missing_flag,payer_code_missing_flag,medical_specialty_missing_flag,weight_missing_flag
0,2278392,8222157,0,Caucasian,Female,[0-10),1,41,0,1,0,0,0,1,5,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1-2 days,0,1-3,1-10,6,25,1,Diabetes,Missing,Missing,Rare,0,1,0,1
1,149190,55629189,0,Caucasian,Female,[10-20),3,59,0,18,0,0,0,9,15,2,0,1,0,0,0,0,0,0,0,1,1,1,1,3-4 days,0,7-9,11-20,1,1,7,Other,Diabetes,Other,Missing,0,1,1,1
2,64410,86047875,0,AfricanAmerican,Female,[20-30),2,11,5,13,2,0,1,6,25,3,3,0,0,0,1,0,1,0,0,0,0,1,0,1-2 days,2-3,4-6,11-20,1,1,7,Other,Diabetes,Supplementary_V,Missing,0,1,1,1
3,500364,82442376,0,Caucasian,Male,[30-40),2,44,1,16,0,0,0,7,35,4,0,1,0,0,0,0,0,0,0,1,1,1,1,1-2 days,0,7-9,11-20,1,1,7,Other,Diabetes,Circulatory,Missing,0,1,1,1
4,16680,42519267,0,Caucasian,Male,[40-50),1,51,0,8,0,0,0,5,45,5,0,0,0,0,0,0,0,0,0,0,1,1,1,1-2 days,0,4-6,1-10,1,1,7,Neoplasms,Neoplasms,Diabetes,Missing,0,1,1,1


In [25]:
model_ready_missingness = pd.DataFrame({
    "column": df_model_ready.columns,
    "missing_count": df_model_ready.isna().sum().values,
    "missing_pct": (df_model_ready.isna().mean().values * 100).round(2)
}).sort_values("missing_pct", ascending=False)

display(model_ready_missingness.head(30))

,column,missing_count,missing_pct
3,race,2273,2.23
0,encounter_id,0,0.00
1,patient_nbr,0,0.00
2,readmitted_30d,0,0.00
4,gender,0,0.00
5,age,0,0.00
6,time_in_hospital,0,0.00
7,num_lab_procedures,0,0.00
8,num_procedures,0,0.00
9,num_medications,0,0.00


In [26]:
excluded_columns = sorted(set(df_fe.columns) - set(model_ready_columns))

excluded_columns_review = pd.DataFrame({
    "excluded_column": excluded_columns,
    "reason": "not_selected_for_primary_model_ready_dataset"
})

display(excluded_columns_review)

,excluded_column,reason
0,A1Cresult,not_selected_for_primary_model_ready_dataset
1,acarbose,not_selected_for_primary_model_ready_dataset
2,acetohexamide,not_selected_for_primary_model_ready_dataset
3,admission_source_id,not_selected_for_primary_model_ready_dataset
4,admission_type_id,not_selected_for_primary_model_ready_dataset
5,change,not_selected_for_primary_model_ready_dataset
6,chlorpropamide,not_selected_for_primary_model_ready_dataset
7,citoglipton,not_selected_for_primary_model_ready_dataset
8,diabetesMed,not_selected_for_primary_model_ready_dataset
9,diag_1,not_selected_for_primary_model_ready_dataset


### Interpretation

The model-ready dataset is intentionally narrower than the cleaned dataset.

It keeps:

- tracking columns
- target
- subgroup review fields
- selected model candidate features

It excludes raw columns that are identifiers, target-derived, overly sparse, high-cardinality, or not selected for the first modeling version.

“Model-ready” here means the dataset has selected candidate modeling columns and required tracking/target fields. It is not yet encoded, imputed, scaled, or split into train/test sets.

## 18. Validation Checks

In [27]:
# Basic safety checks before export.
assert df_model_ready.shape[0] == df.shape[0], "Row count changed unexpectedly."
assert "encounter_id" in df_model_ready.columns, "Missing encounter_id."
assert "patient_nbr" in df_model_ready.columns, "Missing patient_nbr for patient-aware splitting."
assert target_column in df_model_ready.columns, "Missing target column."
assert set(df_model_ready[target_column].dropna().unique()) <= {0, 1}, "Target must be binary."

# These should not appear as predictive features.
for forbidden_feature in ["encounter_id", "patient_nbr", "readmitted"]:
    assert forbidden_feature not in primary_model_features, f"{forbidden_feature} found in model features."

print("Validation checks passed.")
print("Rows:", df_model_ready.shape[0])
print("Columns:", df_model_ready.shape[1])

Validation checks passed.
Rows: 101766
Columns: 44


In [28]:
feature_metadata = feature_list.copy()
feature_metadata["included_in_primary_model"] = True

display(feature_metadata)

,feature,feature_type,included_in_primary_model
0,time_in_hospital,numeric,True
1,num_lab_procedures,numeric,True
2,num_procedures,numeric,True
3,num_medications,numeric,True
4,number_outpatient,numeric,True
5,number_emergency,numeric,True
6,number_inpatient,numeric,True
7,number_diagnoses,numeric,True
8,age_midpoint,numeric,True
9,age_order,numeric,True


## 19. Export Outputs

In [29]:
model_ready_output_path = DATA_PROCESSED / "diabetes_readmission_model_ready.csv"
feature_list_output_path = MODEL_RESULTS / "feature_list.csv"
leakage_review_output_path = MODEL_RESULTS / "leakage_review.csv"
excluded_columns_output_path = MODEL_RESULTS / "excluded_columns_review.csv"
model_ready_missingness_output_path = MODEL_RESULTS / "model_ready_missingness_summary.csv"

df_model_ready.to_csv(model_ready_output_path, index=False)
feature_metadata.to_csv(feature_list_output_path, index=False)
leakage_review.to_csv(leakage_review_output_path, index=False)
excluded_columns_review.to_csv(excluded_columns_output_path, index=False)
model_ready_missingness.to_csv(model_ready_missingness_output_path, index=False)

print("Saved model-ready dataset to:", model_ready_output_path.relative_to(PROJECT_ROOT))
print("Saved feature list to:", feature_list_output_path.relative_to(PROJECT_ROOT))
print("Saved leakage review to:", leakage_review_output_path.relative_to(PROJECT_ROOT))
print("Saved excluded columns review to:", excluded_columns_output_path.relative_to(PROJECT_ROOT))
print("Saved model-ready missingness summary to:", model_ready_missingness_output_path.relative_to(PROJECT_ROOT))

Saved model-ready dataset to: data\processed\diabetes_readmission_model_ready.csv
Saved feature list to: outputs\model_results\feature_list.csv
Saved leakage review to: outputs\model_results\leakage_review.csv
Saved excluded columns review to: outputs\model_results\excluded_columns_review.csv
Saved model-ready missingness summary to: outputs\model_results\model_ready_missingness_summary.csv


## Notebook 03 Summary

This notebook created the model-ready dataset for readmission risk modeling.

Main work completed:

- Defined near-discharge prediction timing
- Confirmed encounter-level modeling unit
- Documented why train/test splitting must be patient-aware
- Created age midpoint and age order features
- Created prior utilization flags and grouped utilization features
- Created encounter complexity features
- Grouped diagnosis codes into broad clinical categories
- Created medication change and diabetes medication features
- Created missingness indicator features
- Grouped rare categories for high-cardinality fields
- Converted operational ID fields into categorical features
- Completed leakage review
- Defined the primary model feature list
- Exported the model-ready dataset

Important modeling decision:

The unit of prediction is the encounter, but train/test splitting should use `patient_nbr` as the grouping variable to avoid patient-level leakage.